# SHROOM RAG E5 - Module 1 Prototype

## Simple retrieval and augmented prompts

Το notebook αυτό αποτελεί το πρώτο, **εκπαιδευτικό** στάδιο της επέκτασης RAG. Δεν τροποποιεί τα πειράματα E1-E4 και δεν αποτελεί ακόμη το τελικό E5.

Στόχοι:

1. Να εξαχθεί η λέξη/φράση ανάμεσα στα `<define>` markers των DM παραδειγμάτων.
2. Να δημιουργηθεί knowledge base μόνο από το **unlabeled model-agnostic training set**.
3. Να ανακτηθούν έως τρεις ορισμοί με exact normalized term matching.
4. Να συγκριθούν ένα baseline prompt και ένα retrieval-augmented prompt.
5. Να επιστρέφεται δομημένο `p_hallucination` από το Gemini και να αποθηκεύεται κάθε κλήση.

### Ασφάλεια και μεθοδολογία

- Το official test set δεν φορτώνεται.
- Τα `label`, `labels` και `p(Hallucination)` δεν μπαίνουν σε query ή prompt.
- Τα API calls είναι απενεργοποιημένα από προεπιλογή.
- Δεν χρησιμοποιούνται ακόμη LangChain, LlamaIndex, FAISS ή vector database.
- Τα αποτελέσματα του μικρού prototype είναι διαγνωστικά και όχι model-selection results.

## Τι κάνουμε με πολύ απλά λόγια

Το `hyp` είναι ο ορισμός που παρήγαγε ένα μοντέλο και θέλουμε να αποφασίσουμε αν είναι hallucination.

1. Βρίσκουμε ποια λέξη ζητείται να οριστεί μέσα στο `src`.
2. Αναζητούμε στο unlabeled training set άλλους διαθέσιμους ορισμούς της ίδιας λέξης.
3. Ρωτάμε το Gemini δύο φορές: πρώτα χωρίς τους ανακτημένους ορισμούς και μετά με αυτούς.
4. Ελέγχουμε αν το πρόσθετο evidence κάνει την εκτίμηση `p_hallucination` πιο σωστή.
5. Αν το RAG βοηθήσει σε μεγαλύτερη και αυστηρή αξιολόγηση, η πιθανότητά του θα προστεθεί αργότερα ως νέο feature στο E3.

Σε αυτό το Module 1 ελέγχουμε μόνο ότι η βασική ιδέα και τα prompts λειτουργούν. Δεν έχουμε δημιουργήσει ακόμη το τελικό E5.


## Πορεία του Module 1

1. Φόρτωση `train.model-agnostic.json` και `val.model-agnostic.json`.
2. Έλεγχος ότι το training corpus είναι unlabeled.
3. Εξαγωγή και κανονικοποίηση των DM terms.
4. Κατασκευή exact-match knowledge base.
5. Σταθερή τυχαία επιλογή 20 DM development examples, χωρίς χρήση labels.
6. Retrieval diagnostics και χειροκίνητος έλεγχος evidence.
7. Κατασκευή baseline/RAG prompts.
8. Προαιρετικές Gemini κλήσεις με checkpoint/resume.
9. Περιγραφική σύγκριση των δύο prompts.

## 1. Imports και σταθερές ρυθμίσεις

In [ ]:
import hashlib
import json
import os
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
PROTOTYPE_N = 20
TOP_K = 3

# Κανένα API call δεν πραγματοποιείται όσο παραμένει False.
RUN_API_CALLS = False

# Στην πρώτη δοκιμή χρησιμοποιούμε μόνο 3 examples. Μετά τον έλεγχο μπορεί να γίνει 20.
MAX_API_SAMPLES = 3

# Επαναχρησιμοποιούμε το μοντέλο που υπάρχει ήδη στην υλοποίηση, αλλά επιτρέπουμε override.
RAG_MODEL = os.environ.get("SHROOM_RAG_MODEL", "gemini-2.5-flash-lite")

DEFAULT_DATA_DIR = Path((__import__('os').environ.get('SHROOM_PROJECT_ROOT', '.') + '/data'))
DATA_DIR = Path(os.environ.get("SHROOM_DATA_DIR", str(DEFAULT_DATA_DIR)))
OUTPUT_DIR = Path(os.environ.get("SHROOM_RAG_OUTPUT_DIR", "results/rag_module1"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = (
    DATA_DIR / "Unlabeled train" / "SHROOM_unlabeled-training-data-v2"
    / "train.model-agnostic.json"
)
DEV_PATH = DATA_DIR / "Dev" / "SHROOM_dev-v2" / "val.model-agnostic.json"
CACHE_PATH = OUTPUT_DIR / "module1_gemini_cache.jsonl"

np.random.seed(RANDOM_STATE)

print("DATA_DIR:", DATA_DIR)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("DEV_PATH exists:", DEV_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("RAG_MODEL:", RAG_MODEL)
print("RUN_API_CALLS:", RUN_API_CALLS)

### Αν τα paths δεν βρεθούν

Στο PowerShell, πριν ανοίξει το Jupyter, μπορεί να οριστεί:

```powershell
$env:SHROOM_DATA_DIR = "data"
```

Δεν αποθηκεύουμε credentials ή προσωπικά paths μέσα στο notebook.

## 2. Φόρτωση μόνο των επιτρεπόμενων datasets

In [ ]:
def load_json_to_df(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Δεν βρέθηκε το αρχείο: {path}\n"
            "Όρισε σωστά τη μεταβλητή περιβάλλοντος SHROOM_DATA_DIR."
        )
    with path.open("r", encoding="utf-8") as stream:
        return pd.DataFrame(json.load(stream))


train_agnostic_df = load_json_to_df(TRAIN_PATH)
dev_agnostic_df = load_json_to_df(DEV_PATH)

print("Unlabeled train shape:", train_agnostic_df.shape)
print("Development shape:", dev_agnostic_df.shape)
print("Train columns:", train_agnostic_df.columns.tolist())
print("Development columns:", dev_agnostic_df.columns.tolist())

In [ ]:
required_text_cols = {"task", "src", "tgt", "hyp"}
missing_train = required_text_cols - set(train_agnostic_df.columns)
missing_dev = required_text_cols - set(dev_agnostic_df.columns)

assert not missing_train, f"Missing train columns: {sorted(missing_train)}"
assert not missing_dev, f"Missing development columns: {sorted(missing_dev)}"

# Η knowledge base πρέπει να προέρχεται αποκλειστικά από unlabeled δεδομένα.
forbidden_training_cols = {"label", "labels", "p(Hallucination)", "p_hallucination"}
present_forbidden = forbidden_training_cols & set(train_agnostic_df.columns)
assert not present_forbidden, (
    "Το training corpus δεν είναι unlabeled. Βρέθηκαν: "
    f"{sorted(present_forbidden)}"
)

assert len(dev_agnostic_df) == 499, (
    f"Expected 499 development rows, found {len(dev_agnostic_df)}"
)

print("Dataset safety checks passed.")

## 3. Εξαγωγή του `<define>` term

Στο SHROOM εμφανίζονται τουλάχιστον δύο μορφές:

- `<define>word</define>`
- `<define> word <define>`

Η συνάρτηση υποστηρίζει και τις δύο. Αν δεν υπάρχει σαφές ζεύγος markers, επιστρέφει `None` αντί να μαντεύει.

In [ ]:
CLOSED_DEFINE_PATTERN = re.compile(
    r"<define>\s*(.*?)\s*</define>",
    flags=re.IGNORECASE | re.DOTALL,
)
OPEN_DEFINE_MARKER = re.compile(r"<define>", flags=re.IGNORECASE)


def clean_text(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def extract_define_term(source_text) -> str | None:
    text = clean_text(source_text)
    if not text:
        return None

    closed_match = CLOSED_DEFINE_PATTERN.search(text)
    if closed_match:
        candidate = closed_match.group(1).strip()
        return candidate or None

    parts = OPEN_DEFINE_MARKER.split(text)
    if len(parts) >= 3:
        candidate = parts[1].strip()
        return candidate or None

    return None


def normalize_term(term) -> str | None:
    text = clean_text(term)
    if not text:
        return None

    text = unicodedata.normalize("NFKC", text).lower()
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s*-\s*", "-", text)
    text = re.sub(r"[^\w\s\-']+", " ", text, flags=re.UNICODE)
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def valid_definition(value) -> bool:
    text = clean_text(value)
    return bool(text and re.search(r"[A-Za-z0-9]", text))

In [ ]:
# Offline unit tests: δεν χρειάζονται dataset ή API.
extractor_cases = {
    "A <define> weasel-like </define> animal": "weasel-like",
    "A <define> sheath - knife <define> used by sailors": "sheath - knife",
    "No marker here": None,
    "": None,
}

for source, expected in extractor_cases.items():
    actual = extract_define_term(source)
    assert actual == expected, (source, expected, actual)

assert normalize_term(" Sheath - Knife ") == "sheath knife"
assert normalize_term("WEASEL–LIKE") == "weasel like"
assert valid_definition("A short moment.")
assert not valid_definition(".")

print("Offline extraction tests passed.")

## 4. Knowledge base από το unlabeled DM training corpus

In [ ]:
def build_dm_knowledge_base(train_df: pd.DataFrame) -> pd.DataFrame:
    dm = train_df.loc[train_df["task"].astype(str).eq("DM")].copy()
    dm = dm.reset_index().rename(columns={"index": "evidence_id"})

    dm["term_raw"] = dm["src"].map(extract_define_term)
    dm["term_key"] = dm["term_raw"].map(normalize_term)
    dm["definition"] = dm["tgt"].map(clean_text)
    dm["source_context"] = dm["src"].map(clean_text)

    kb = dm.loc[
        dm["term_key"].notna() & dm["definition"].map(valid_definition),
        ["evidence_id", "term_raw", "term_key", "definition", "source_context"],
    ].copy()

    kb = kb.drop_duplicates(subset=["term_key", "definition"]).reset_index(drop=True)
    return kb


dm_kb = build_dm_knowledge_base(train_agnostic_df)

print("DM training rows:", int(train_agnostic_df["task"].eq("DM").sum()))
print("Valid unique knowledge-base rows:", len(dm_kb))
print("Unique normalized terms:", dm_kb["term_key"].nunique())
display(dm_kb.head())

In [ ]:
def retrieve_exact_definitions(
    term,
    knowledge_base: pd.DataFrame,
    top_k: int = TOP_K,
) -> list[dict]:
    term_key = normalize_term(term)
    if term_key is None:
        return []

    matches = knowledge_base.loc[knowledge_base["term_key"].eq(term_key)].head(top_k)
    return matches.to_dict(orient="records")


# Μικρό offline smoke test με τεχνητή knowledge base.
toy_kb = pd.DataFrame([
    {
        "evidence_id": 1,
        "term_raw": "sheath-knife",
        "term_key": "sheath knife",
        "definition": "A knife carried in a sheath.",
        "source_context": "A sailor carried a sheath-knife.",
    }
])
toy_results = retrieve_exact_definitions("sheath - knife", toy_kb)
assert len(toy_results) == 1
assert toy_results[0]["evidence_id"] == 1
print("Offline exact-retrieval test passed.")

## 5. Σταθερό prototype sample από το DM development set

Η επιλογή γίνεται τυχαία με `random_state=42` και **δεν** χρησιμοποιεί label ή `p(Hallucination)`. Τα labels θα χρησιμοποιηθούν μόνο μετά την παραγωγή, για περιγραφική αξιολόγηση.

In [ ]:
dm_dev = dev_agnostic_df.loc[dev_agnostic_df["task"].astype(str).eq("DM")].copy()
dm_dev = dm_dev.reset_index().rename(columns={"index": "development_index"})

n_prototype = min(PROTOTYPE_N, len(dm_dev))
prototype_df = (
    dm_dev.sample(n=n_prototype, random_state=RANDOM_STATE)
    .sort_values("development_index")
    .reset_index(drop=True)
)

prototype_df["define_term"] = prototype_df["src"].map(extract_define_term)
prototype_df["define_term_key"] = prototype_df["define_term"].map(normalize_term)
prototype_df["evidence"] = prototype_df["define_term"].map(
    lambda term: retrieve_exact_definitions(term, dm_kb, top_k=TOP_K)
)
prototype_df["evidence_count"] = prototype_df["evidence"].map(len)

print("Prototype rows:", len(prototype_df))
print("Successful term extraction:", int(prototype_df["define_term"].notna().sum()))
print("Rows with retrieved evidence:", int(prototype_df["evidence_count"].gt(0).sum()))
print("Exact-retrieval coverage:", f"{prototype_df['evidence_count'].gt(0).mean():.1%}")

display(
    prototype_df[
        ["development_index", "define_term", "evidence_count", "src", "tgt", "hyp"]
    ]
)

In [ ]:
def evidence_preview(row) -> str:
    lines = []
    for item in row["evidence"]:
        lines.append(
            f"ID={item['evidence_id']} | term={item['term_raw']} | "
            f"definition={item['definition']}"
        )
    return "\n".join(lines) if lines else "NO EVIDENCE"


# Για το prompt comparison χρειαζόμαστε examples στα οποία το RAG
# έχει πράγματι ανακτήσει evidence. Διαφορετικά baseline και RAG prompt
# θα περιείχαν ουσιαστικά την ίδια πληροφορία.
evidence_prototype_df = (
    prototype_df.loc[prototype_df["evidence_count"].gt(0)]
    .copy()
    .reset_index(drop=True)
)

if evidence_prototype_df.empty:
    raise ValueError(
        "No prototype examples have retrieved evidence. "
        "Do not run Gemini calls; improve retrieval first."
    )

print("Examples with retrieved evidence:", len(evidence_prototype_df))

# Εμφανίζουμε όλα τα evidence-bearing examples για χειροκίνητο έλεγχο
# πριν επιτραπούν οποιεσδήποτε API κλήσεις.
for _, row in evidence_prototype_df.iterrows():
    print("=" * 80)
    print("Development index:", row["development_index"])
    print("Extracted term:", row["define_term"])
    print("Reference:", row["tgt"])
    print("Hypothesis:", row["hyp"])
    print("Retrieved evidence:\n" + evidence_preview(row))

### Πύλη απόφασης πριν από Gemini

Ελέγχουμε χειροκίνητα τα παραπάνω αποτελέσματα. Δεν ενεργοποιούμε API calls αν:

- το term extraction αποτυγχάνει συχνά,
- τα definitions ανήκουν σε διαφορετική λέξη,
- το evidence είναι άσχετο με το context,
- η exact-retrieval coverage είναι πολύ χαμηλή.

Η χαμηλή κάλυψη δεν είναι αποτυχία. Είναι το αναμενόμενο κίνητρο για BM25/semantic retrieval στο Module 2.

## 6. Baseline και retrieval-augmented prompts

In [ ]:
SYSTEM_INSTRUCTION = """You are a careful evaluator of hallucinations in natural language generation.
Evaluate semantic support, not fluency. Follow the requested JSON schema exactly."""


def build_baseline_prompt(*, task: str, source: str, reference: str, hypothesis: str) -> str:
    return f"""
Task: {clean_text(task)}

Source:
{clean_text(source)}

Primary reference:
{clean_text(reference)}

AI-generated hypothesis:
{clean_text(hypothesis)}

Question:
Does the hypothesis contain meaning that is unsupported by the primary reference?

Rules:
- The primary reference is the main semantic ground truth.
- Do not judge writing style or fluency.
- Do not use outside knowledge.
- Return JSON only.
- p_hallucination must be between 0 and 1.
- evidence_ids must be an empty array because no retrieved evidence was provided.
""".strip()


def format_evidence(evidence: list[dict]) -> str:
    if not evidence:
        return "No retrieved evidence was available."

    blocks = []
    for item in evidence:
        blocks.append(
            "\n".join([
                f"Evidence ID: {int(item['evidence_id'])}",
                f"Defined term: {clean_text(item['term_raw'])}",
                f"Definition: {clean_text(item['definition'])}",
                f"Usage context: {clean_text(item['source_context'])}",
            ])
        )
    return "\n\n".join(blocks)


def build_rag_prompt(
    *,
    task: str,
    source: str,
    reference: str,
    hypothesis: str,
    evidence: list[dict],
) -> str:
    return f"""
Task: {clean_text(task)}

Source:
{clean_text(source)}

Primary reference:
{clean_text(reference)}

Retrieved evidence:
{format_evidence(evidence)}

AI-generated hypothesis:
{clean_text(hypothesis)}

Question:
Does the hypothesis contain meaning that is unsupported by the primary reference and relevant retrieved evidence?

Rules:
- The primary reference is the main semantic ground truth.
- Retrieved evidence is secondary and may contain a different word sense.
- Ignore evidence that is irrelevant to the current source context.
- Do not judge writing style or fluency.
- Do not add outside knowledge.
- Return JSON only.
- p_hallucination must be between 0 and 1.
- evidence_ids may contain only IDs that materially supported the decision.
""".strip()

In [ ]:
# Δημιουργούμε prompts μόνο από whitelisted text fields και evidence.
example_row = evidence_prototype_df.iloc[0]

baseline_prompt = build_baseline_prompt(
    task=example_row["task"],
    source=example_row["src"],
    reference=example_row["tgt"],
    hypothesis=example_row["hyp"],
)
rag_prompt = build_rag_prompt(
    task=example_row["task"],
    source=example_row["src"],
    reference=example_row["tgt"],
    hypothesis=example_row["hyp"],
    evidence=example_row["evidence"],
)

print("BASELINE PROMPT\n", "-" * 80, "\n", baseline_prompt)
print("\nRAG PROMPT\n", "-" * 80, "\n", rag_prompt)

## 7. Structured output και validation

Το Gemini πρέπει να επιστρέφει μόνο:

```json
{
  "p_hallucination": 0.75,
  "label": "Hallucination",
  "reason": "Brief evidence-grounded explanation.",
  "evidence_ids": [123]
}
```

Η πιθανότητα αποτελεί το βασικό υποψήφιο RAG feature για το μελλοντικό E5.

In [ ]:
RAG_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "p_hallucination": {"type": "number"},
        "label": {
            "type": "string",
            "enum": ["Hallucination", "Not Hallucination"],
        },
        "reason": {"type": "string"},
        "evidence_ids": {
            "type": "array",
            "items": {"type": "integer"},
        },
    },
    "required": ["p_hallucination", "label", "reason", "evidence_ids"],
}


def validate_rag_response(payload: dict, *, mode: str, allowed_evidence_ids: set[int]) -> dict:
    required = {"p_hallucination", "label", "reason", "evidence_ids"}
    missing = required - set(payload)
    if missing:
        raise ValueError(f"Missing response fields: {sorted(missing)}")

    probability = float(payload["p_hallucination"])
    if not 0.0 <= probability <= 1.0:
        raise ValueError("p_hallucination must be in [0, 1]")

    label = str(payload["label"])
    if label not in {"Hallucination", "Not Hallucination"}:
        raise ValueError(f"Unexpected label: {label}")

    evidence_ids = [int(item) for item in payload["evidence_ids"]]
    if mode == "baseline" and evidence_ids:
        raise ValueError("Baseline response must not cite evidence IDs")
    if not set(evidence_ids).issubset(allowed_evidence_ids):
        raise ValueError("Response cited an evidence ID that was not retrieved")

    return {
        "p_hallucination": probability,
        "label": label,
        "reason": clean_text(payload["reason"]),
        "evidence_ids": evidence_ids,
        "label_probability_consistent": (
            (probability >= 0.5 and label == "Hallucination")
            or (probability < 0.5 and label == "Not Hallucination")
        ),
    }

## 8. Credentials χωρίς hardcoded secrets

Πριν από API calls ορίζονται στο PowerShell:

```powershell
$env:GOOGLE_APPLICATION_CREDENTIALS = "external-artifacts/local-bd4c253761d6/service-account.json"
$env:GOOGLE_CLOUD_PROJECT = "your-project-id"
$env:GOOGLE_CLOUD_LOCATION = "us-central1"
```

Το JSON key δεν ανεβαίνει σε notebook, GitHub ή συνομιλία.

In [ ]:
def create_gemini_client():
    from google import genai
    from google.oauth2 import service_account

    credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    project_id = os.environ.get("GOOGLE_CLOUD_PROJECT")
    location = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

    if not credentials_path:
        raise EnvironmentError("GOOGLE_APPLICATION_CREDENTIALS is not set")
    if not project_id:
        raise EnvironmentError("GOOGLE_CLOUD_PROJECT is not set")

    credential_file = Path(credentials_path)
    if not credential_file.exists():
        raise FileNotFoundError(f"Credential file not found: {credential_file}")

    credentials = service_account.Credentials.from_service_account_file(
        credential_file,
        scopes=["https://www.googleapis.com/auth/cloud-platform"],
    )
    return genai.Client(
        vertexai=True,
        project=project_id,
        location=location,
        credentials=credentials,
    )

## 9. Checkpoint/resume και Gemini call

In [ ]:
def prompt_hash(prompt: str) -> str:
    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()


def load_cache(path: Path) -> dict[tuple, dict]:
    cache = {}
    if not path.exists():
        return cache

    with path.open("r", encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            record = json.loads(line)
            key = (
                int(record["development_index"]),
                record["mode"],
                record["model"],
                record["prompt_hash"],
            )
            cache[key] = record
    return cache


def append_cache(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")


def call_gemini_verifier(client, *, prompt: str, mode: str, evidence: list[dict]) -> dict:
    from google.genai import types

    response = client.models.generate_content(
        model=RAG_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.0,
            max_output_tokens=512,
            response_mime_type="application/json",
            response_schema=RAG_RESPONSE_SCHEMA,
        ),
    )

    if not getattr(response, "text", None):
        raise ValueError("Gemini returned an empty response")

    payload = json.loads(response.text)
    allowed_ids = {int(item["evidence_id"]) for item in evidence}
    return validate_rag_response(
        payload,
        mode=mode,
        allowed_evidence_ids=allowed_ids,
    )

In [ ]:
def run_module1_api_experiment(
    prototype: pd.DataFrame,
    *,
    max_samples: int = MAX_API_SAMPLES,
) -> pd.DataFrame:
    if not RUN_API_CALLS:
        print("RUN_API_CALLS=False. No Gemini calls were made.")
        return pd.DataFrame()

    client = create_gemini_client()
    cache = load_cache(CACHE_PATH)
    new_records = []

    for _, row in prototype.head(max_samples).iterrows():
        for mode in ("baseline", "rag"):
            evidence = [] if mode == "baseline" else row["evidence"]

            if mode == "baseline":
                prompt = build_baseline_prompt(
                    task=row["task"],
                    source=row["src"],
                    reference=row["tgt"],
                    hypothesis=row["hyp"],
                )
            else:
                prompt = build_rag_prompt(
                    task=row["task"],
                    source=row["src"],
                    reference=row["tgt"],
                    hypothesis=row["hyp"],
                    evidence=evidence,
                )

            current_hash = prompt_hash(prompt)
            key = (int(row["development_index"]), mode, RAG_MODEL, current_hash)

            if key in cache:
                print("Cache hit:", key[:3])
                continue

            last_error = None
            for attempt in range(1, 3):
                try:
                    result = call_gemini_verifier(
                        client,
                        prompt=prompt,
                        mode=mode,
                        evidence=evidence,
                    )
                    record = {
                        "development_index": int(row["development_index"]),
                        "mode": mode,
                        "model": RAG_MODEL,
                        "prompt_hash": current_hash,
                        "retrieved_evidence_ids": [
                            int(item["evidence_id"]) for item in evidence
                        ],
                        **result,
                    }
                    append_cache(CACHE_PATH, record)
                    cache[key] = record
                    new_records.append(record)
                    print("Saved:", key[:3])
                    last_error = None
                    break
                except Exception as exc:
                    last_error = exc
                    print(f"Attempt {attempt}/2 failed:", type(exc).__name__, str(exc)[:200])
                    if attempt < 2:
                        time.sleep(3)

            if last_error is not None:
                print("Skipped after retries:", key[:3])

            time.sleep(1)

    return pd.DataFrame(new_records)


# Οι πρώτες κλήσεις γίνονται μόνο σε examples με πραγματικό evidence.
new_api_records_df = run_module1_api_experiment(
    evidence_prototype_df,
    max_samples=min(MAX_API_SAMPLES, len(evidence_prototype_df)),
)
display(new_api_records_df)

### Πώς ενεργοποιούνται με ασφάλεια οι πρώτες κλήσεις

1. Ελέγχουμε χειροκίνητα τα retrieved definitions και τα δύο prompts.
2. Ορίζουμε τα credential environment variables εκτός notebook.
3. Αλλάζουμε μόνο `RUN_API_CALLS = True`.
4. Κρατάμε `MAX_API_SAMPLES = 3`.
5. Εκτελούμε και ελέγχουμε τα έξι αποτελέσματα: 3 baseline + 3 RAG, όλα πάνω σε examples που έχουν retrieved evidence.
6. Μόνο μετά αυξάνουμε στα 20.

## 10. Περιγραφική αξιολόγηση του prototype

In [ ]:
def evaluate_cached_prototype(
    prototype: pd.DataFrame,
    cache_path: Path = CACHE_PATH,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not cache_path.exists():
        print("No cache exists yet. Run the approved API prototype first.")
        return pd.DataFrame(), pd.DataFrame()

    records = []
    with cache_path.open("r", encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                records.append(json.loads(line))

    responses = pd.DataFrame(records)
    if responses.empty:
        return responses, pd.DataFrame()

    # Ground truth προστίθεται μόνο μετά την παραγωγή, ποτέ στο prompt.
    truth_cols = ["development_index", "label", "p(Hallucination)"]
    merged = responses.merge(prototype[truth_cols], on="development_index", how="inner")
    merged["y_true_bin"] = merged["label_y"].map(
        {"Hallucination": 1, "Not Hallucination": 0}
    ) if "label_y" in merged.columns else merged["label"].map(
        {"Hallucination": 1, "Not Hallucination": 0}
    )

    prediction_label_col = "label_x" if "label_x" in merged.columns else "label"
    merged["y_pred_bin"] = merged[prediction_label_col].map(
        {"Hallucination": 1, "Not Hallucination": 0}
    )

    summary_rows = []
    for mode, group in merged.groupby("mode"):
        accuracy = float((group["y_true_bin"] == group["y_pred_bin"]).mean())
        spearman = group["p(Hallucination)"].corr(
            group["p_hallucination"], method="spearman"
        ) if len(group) >= 2 else np.nan
        summary_rows.append({
            "mode": mode,
            "samples": len(group),
            "accuracy_descriptive_only": accuracy,
            "spearman_descriptive_only": spearman,
        })

    summary = pd.DataFrame(summary_rows)
    print("These small-sample metrics are descriptive only, not model selection.")
    return merged, summary


prototype_results_df, prototype_summary_df = evaluate_cached_prototype(prototype_df)
display(prototype_summary_df)

## 11. Κριτήρια ολοκλήρωσης Module 1

Το Module 1 θεωρείται ολοκληρωμένο όταν:

- [ ] Το extraction λειτουργεί στις δύο γνωστές μορφές `<define>`.
- [ ] Η knowledge base προέρχεται μόνο από unlabeled train.
- [ ] Έχει καταγραφεί η exact-retrieval coverage στα 20 DM examples.
- [ ] Έχουν ελεγχθεί χειροκίνητα τουλάχιστον 5 retrieved results.
- [ ] Τα baseline και RAG prompts δεν περιλαμβάνουν gold labels.
- [ ] Οι πρώτες 3+3 Gemini κλήσεις επιστρέφουν έγκυρο JSON.
- [ ] Το checkpoint/resume αποτρέπει επαναλαμβανόμενα API calls.
- [ ] Έχουν καταγραφεί παραδείγματα όπου το evidence βοήθησε ή μπέρδεψε τον verifier.

### Επόμενο βήμα: Module 2

Μετά την ολοκλήρωση των παραπάνω θα προσθέσουμε BM25, semantic retrieval, metadata filtering και retrieval evaluation. Δεν προχωράμε ακόμη σε τελικό E5 ή official test evaluation.